## example notebook for parsing tcrs

In [ ]:
import sys
from pathlib import Path
import pandas as pd
import os
import numpy as np
from collections import defaultdict
import glob as glob

sys.path.append("/TCR_toolbox/tcr_toolbox")
sys.path.append("/TCR_toolbox")

from tcr_toolbox.tcr_parsing.parser_utils import calculate_clonotype_frequencies, drop_clonotype_duplicates
from tcr_toolbox.utils.plot_utils import plot_10x_tcr_umi_counts_hist, plot_clonotype_frequency

from tcr_toolbox.tcr_parsing.parse_PRIVATE.tcr_10x_contig_parser import filter_meta_10x_df_on_tcr_umi_counts, merge_combined_meta_df_with_tcr_contig_dfs, parse_10x_contig_list, parse_10x_from_contig

In [ ]:
filtered_contig_base_path = '/tcr_seq_data/DFCI_datasets/OV/cellranger_outputs'
project_dir_path = os.getcwd()
outputs = Path(project_dir_path) / 'outputs'
os.makedirs(outputs, exist_ok=True)
meta_path = '/tcr_seq_data/DFCI_datasets/OV/analysis_notebooks/outputs/meta_df.csv'

In [ ]:
list_of_config_paths = glob.glob(os.path.join(filtered_contig_base_path, "**", "vdj_t", "filtered_contig_annotations.csv"),recursive=True)

list_of_patients = [path.split('/')[-3] for path in list_of_config_paths]

for patient, path in zip(list_of_patients, list_of_config_paths):
    print(patient, path)

In [8]:
dual_chain_umi_count_ratio_threshold = 0.4

In [9]:
contig_df = parse_10x_contig_list(
    gem_10x_run_name_list=list_of_patients,
    contig_file_list=list_of_config_paths,
    seq_run_dir=project_dir_path,
    outs_dir=outputs,
    meta_data_column_list=None,
    filter_on_quality=True,
    keep_highest_read_only=False,
    remove_dual_alpha_and_dual_beta_barcodes=False,
    remove_dual_beta_barcodes=False,
    remove_dual_alpha_barcodes=False,
    remove_single_chain_barcodes=True,
    remove_only_alpha_or_beta_chain_barcodes=True,
    dual_chain_umi_count_ratio_threshold=dual_chain_umi_count_ratio_threshold,
    threads=8)

# contig_df.to_csv(outputs / "contig_df.csv", index=False)

## merge with meta

In [ ]:
meta_df = pd.read_csv(meta_path, index_col=0)
combined_meta_df = merge_combined_meta_df_with_tcr_contig_dfs(combined_contig_df=contig_df, 
                                                              combined_meta_df=meta_df)

In [14]:
# plot UMI counts alpha & beta chain distributions before filtering

umi_count_quantile_threshold = 0.025

plot_10x_tcr_umi_counts_hist(
    combined_meta_df=combined_meta_df,
    save_dir=outputs,
    fig_name="before_filtering",
    bin_max=150,
    bin_size=2,
    ylim_max=0.2,
    umi_count_quantile_threshold=umi_count_quantile_threshold)

In [15]:
combined_meta_df = filter_meta_10x_df_on_tcr_umi_counts(meta_10x_df=combined_meta_df, sample_col="data_origin", umi_count_quantile_threshold=umi_count_quantile_threshold)

Shape combined_meta_df before UMI count quantile filtering: 12350
Shape combined_meta_df after UMI count quantile filtering: 12121


In [17]:
plot_10x_tcr_umi_counts_hist(combined_meta_df=combined_meta_df, 
                             save_dir=outputs, 
                             fig_name="after_filtering", 
                             bin_max=200, 
                             bin_size=1, 
                             ylim_max=0.2)

In [ ]:
combined_meta_df = calculate_clonotype_frequencies(
    tcr_meta_df=combined_meta_df,
    donor_col="patient",
    detected_chains_col="detected_chains",
    clonotype_nt_col="clonotype_nt",
    clonotype_aa_col="clonotype_aa",
    clonotype_count_nt_col="clonotype_count_nt",
    clonotype_count_aa_col="clonotype_count_aa",
    clonotype_freq_nt_col="clonotype_frequency_nt",
    clonotype_freq_aa_col="clonotype_frequency_aa")

In [ ]:
unique_combined_meta_df = drop_clonotype_duplicates(
    tcr_meta_df=combined_meta_df,
    also_drop_clonotype_beta_aa_duplicates=False,
    donor_col="patient",
    clonotype_freq_col="clonotype_frequency_aa",
    clonotype_col="unique_clonotype_aa",
    # clonotype_beta_col = "unique_clonotype_beta_aa",
    clonotype_shared_with_col="shared_with",)

# unique_combined_meta_df.loc[:, 'tumor_type'] = 'OV'

# unique_combined_meta_df.to_csv(outputs / "OV_unique_combined_meta_df.csv", index=False)

Number of TCRs before dropping duplicate clonotypes: 12121
Number of TCRs after dropping duplicate clonotypes: 7761
